# Overview of the Action Composer

The BioJEPA-AC model is designed to support perturbations of different modalities (DNA, Protein, Chemical) with many different modes (crispri, crispra, overexpression,knockout,inhibitor, agonist, degrader, binder). Since these modalities can be of varied lengths and in some cases we might just have a perturbation target, or the perturbation itself, we use an action composer to create a unified representation of the perturbations we may use.

The ActionComposer is a dual-pathway multi-layer perceptron (MLP) encoder that projects sequence and target embeddings into a shared latent space via separate linear projections, fuses them additively, and applies FiLM conditioning from a learned mode embedding to encode the perturbation mechanism. 

In [1]:
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import math

## Data Prep

We'll start by a simple data prep. Here we'll use a single example in the batch, but the example will have 3 different perturbations:
1. A CRISPRi perturbation where we know the sgRNA sequence (DNA) and the target protein encoding gene (AA)
2. A CRISPRa perturbation where we just know the target protein (AA) 
3. A drug based perturbation where we know the chemical sequence (SMILES) and the target protein (AA)

This example would be a cell that has had all 3 of these perturbations applied.  This is purely for example purposes to show the flexibility of our model. 

In [2]:
batch = 1
n_perts = 3 # number of perturbations per cell, in this example 3

SEED = 1337
torch.manual_seed(SEED)
np.random.seed(SEED)

**Data Encoding**  in our model, our data prep would involve getting the raw sequences (DNA, SMILES, or Amino Acids) and then passing them through a foundation model to create an embedding representation of each component. In our exmaple we'll jump straight to embeddings.  

In [5]:
torch.from_numpy(np.round(np.random.uniform(1, 5, size=(batch, 2)), 0)).float()

NameError: name 'num_genes' is not defined

In [ ]:
seq_emb = torch.randn(B, N_pert, 1536)        # padded to max seq dim
target_emb = torch.randn(B, N_pert, 320)       # ESM-2 protein embeddings

modality_ids = torch.tensor([[0, 0, 2]])        # DNA, DNA, chemical
mode_ids = torch.tensor([[0, 1, 4]])            # crispri, crispra, inhibitor
has_seq = torch.tensor([[True, False, True]])
has_target = torch.tensor([[True, True, True]])
pert_mask = torch.tensor([[True, True, True]])   # all 3 are real perturbations